# DataGuide → korea_fs_data_from_DG 적재

## 용도
DataGuide에서 다운받은 `FCFF_RIM_DATA.xlsx` (KS) / `FCFF_RIM_KQ_DATA.xlsx` (KQ) 파일을
wide format → long format으로 변환해 MariaDB의 `korea_fs_data_from_DG` 테이블에 적재한다.

## 실행 순서
1. **Cell 0** — 라이브러리 로드 및 설정
2. **Cell 1** — (최초 1회) `mode='initial'` 실행: 기존 데이터 TRUNCATE 후 새로 적재
3. **Cell 2** — (이후) `mode='update'` 실행: 기존 유지, upsert만 수행
4. **Cell 3** — 적재 검증

## 중복 방지 메커니즘
- PK = `(date, ticker, item_code)` → DB 레벨에서 중복 물리적 차단
- `ON DUPLICATE KEY UPDATE` → 같은 PK 재입력 시 값만 업데이트
- `created_at`은 최초 INSERT 시각, `updated_at`은 변경 시각 자동 기록

In [5]:
# ==========================================================
# Cell 0: 라이브러리 로드 및 설정
# ==========================================================
import sys
from pathlib import Path

# ---- 1. 프로젝트 루트 자동 탐색 (기존 DATA 폴더 기반 모듈 import 용) ----
def add_repo_path():
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / 'DATA').exists():
            if str(parent) not in sys.path:
                sys.path.insert(0, str(parent))
            return parent
    raise FileNotFoundError('DATA 폴더를 찾을 수 없습니다')

PROJECT_ROOT = add_repo_path()
print(f'Project root: {PROJECT_ROOT}')

# loader 모듈이 있는 폴더(현재 노트북과 같은 폴더)를 sys.path에 추가
sys.path.insert(0, str(Path.cwd()))

from Korea_Market.collection.dataguide_fs_loader import load_dataguide_to_db
from DATA.stock_invest_function import get_db_host


# ---- 2. DataGuide 엑셀 폴더 경로 (2대 PC 자동 감지) ----
EXCEL_DIR_CANDIDATES = [
    # 데스크탑 (user=82108)
    Path(r"C:\Users\82108\OneDrive\INVESTMENT\한국주식\FCFF_RIM_재무데이터"),
    # 노트북 (user=Hoyoung_Park) — 실제 경로 다르면 여기에 맞춰 수정
    Path(r"C:\Users\Hoyoung_Park\OneDrive\INVESTMENT\한국주식\FCFF_RIM_재무데이터"),
]

EXCEL_DIR = None
for candidate in EXCEL_DIR_CANDIDATES:
    if candidate.exists():
        EXCEL_DIR = candidate
        break

if EXCEL_DIR is None:
    raise FileNotFoundError(
        f"DataGuide 엑셀 폴더를 찾을 수 없습니다.\n"
        f"시도한 경로:\n" + "\n".join(f"  - {c}" for c in EXCEL_DIR_CANDIDATES)
    )

print(f'Excel dir  : {EXCEL_DIR}')


# ---- 3. 파일 경로 설정 ----
file_configs = [
    {'path': EXCEL_DIR / 'FCFF_RIM_DATA.xlsx',    'market': 'KS'},
    {'path': EXCEL_DIR / 'FCFF_RIM_KQ_DATA_20260601.xlsx', 'market': 'KQ'},
]

print('\n[파일 존재 확인]')
for cfg in file_configs:
    exists = '✓' if cfg['path'].exists() else '✗ NOT FOUND'
    size_kb = (cfg['path'].stat().st_size // 1024) if cfg['path'].exists() else 0
    print(f"  {exists} {cfg['path'].name}  ({size_kb:,} KB)")


# ---- 4. DB 연결 정보 ----
db_info = {
    'host':     get_db_host(),
    'port':     3307,
    'user':     'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar',
}
print(f"\nDB host: {db_info['host']}")

Project root: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
Excel dir  : C:\Users\82108\OneDrive\INVESTMENT\한국주식\FCFF_RIM_재무데이터

[파일 존재 확인]
  ✓ FCFF_RIM_DATA.xlsx  (15,743 KB)
  ✓ FCFF_RIM_KQ_DATA_20260601.xlsx  (9,104 KB)

DB host: 192.168.0.230


In [6]:
# ==========================================================
# Cell 1: [최초 1회] 기존 데이터 TRUNCATE 후 새로 적재
# ==========================================================
# ⚠️ 이 셀은 기존 korea_fs_data_from_DG 데이터를 전부 삭제합니다.
#    한 번만 실행하세요. 이후에는 Cell 2 (update 모드)를 사용하세요.
#
# 예상 소요시간: 약 3~5분 (370만 rows)

# total_rows = load_dataguide_to_db(
#     file_configs=file_configs,
#     db_info=db_info,
#     mode='initial',   # ← TRUNCATE 후 적재
# )
# print(f'\n✓ 최초 적재 완료: {total_rows:,} rows')

In [7]:
# ==========================================================
# Cell 2: [평상시] 업데이트 모드 (TRUNCATE 없음)
# ==========================================================
# 같은 (date, ticker, item_code) 조합은 값만 최신화되고,
# 새로운 조합만 INSERT 됩니다. 안심하고 반복 실행 가능합니다.

total_rows = load_dataguide_to_db(
    file_configs=file_configs,
    db_info=db_info,
    mode='update',   # ← upsert만, 기존 데이터 유지
)
print(f'\n✓ 업데이트 완료: {total_rows:,} rows 처리')

2026-06-01 16:55:26 [INFO] ======================================================================
2026-06-01 16:55:26 [INFO] MODE: UPDATE — 기존 데이터 유지, upsert만 실행
2026-06-01 16:55:26 [INFO] ======================================================================
2026-06-01 16:55:26 [INFO] 테이블 확인/생성 완료: korea_fs_data_from_DG
2026-06-01 16:55:26 [INFO] 
[파일 시작] FCFF_RIM_DATA.xlsx (market=KS)
2026-06-01 16:55:26 [INFO] 엑셀 로드 시작: FCFF_RIM_DATA.xlsx (market=KS)
2026-06-01 16:55:26 [INFO]   시트 'IS' 파싱 중... (rows=81, cols=8669)
2026-06-01 16:55:29 [INFO] [KS/IS] 데이터 컬럼 수: 8668
2026-06-01 16:55:30 [INFO]   ... 5,000 rows upserted
2026-06-01 16:55:30 [INFO]   ... 10,000 rows upserted
2026-06-01 16:55:31 [INFO]   ... 15,000 rows upserted
2026-06-01 16:55:31 [INFO]   ... 20,000 rows upserted
2026-06-01 16:55:32 [INFO]   ... 25,000 rows upserted
2026-06-01 16:55:32 [INFO]   ... 30,000 rows upserted
2026-06-01 16:55:33 [INFO]   ... 35,000 rows upserted
2026-06-01 16:55:34 [INFO]   ... 40,000 rows upse


✓ 업데이트 완료: 1,706,264 rows 처리


In [9]:
# ==========================================================
# Cell 3: 적재 검증
# ==========================================================
import pymysql
import pandas as pd

conn = pymysql.connect(
    host=db_info['host'], port=db_info['port'],
    user=db_info['user'], password=db_info['password'],
    database=db_info['database'], charset='utf8mb4',
)

try:
    print('=' * 70)
    print('korea_fs_data_from_DG 적재 검증')
    print('=' * 70)

    # ---- 1. 전체 규모
    df1 = pd.read_sql('''
        SELECT COUNT(*) AS total_rows,
               COUNT(DISTINCT ticker) AS unique_tickers,
               COUNT(DISTINCT item_code) AS unique_indicators,
               MIN(date) AS min_date,
               MAX(date) AS max_date
        FROM korea_fs_data_from_DG
    ''', conn)
    print('\n[1] 전체 규모')
    print(df1.to_string(index=False))

    # ---- 2. 시장/시트별 분포
    df2 = pd.read_sql('''
        SELECT market, sj_div,
               COUNT(*) AS row_cnt,
               COUNT(DISTINCT ticker) AS ticker_cnt,
               COUNT(DISTINCT item_code) AS indicator_cnt
        FROM korea_fs_data_from_DG
        GROUP BY market, sj_div
        ORDER BY market, sj_div
    ''', conn)
    print('\n[2] 시장/시트별 분포')
    print(df2.to_string(index=False))

    # ---- 3. 중복 PK 검사
    df3 = pd.read_sql('''
        SELECT COUNT(*) AS dup_cnt FROM (
            SELECT date, ticker, item_code, COUNT(*) AS c
            FROM korea_fs_data_from_DG
            GROUP BY date, ticker, item_code
            HAVING c > 1
        ) dup
    ''', conn)
    print(f'\n[3] 중복 PK 검사: {df3["dup_cnt"].values[0]}개 (0이어야 정상)')

    # ---- 4. 주요 기업 샘플
    df4 = pd.read_sql('''
        SELECT date, ticker, company_name, indicator, value
        FROM korea_fs_data_from_DG
        WHERE ticker = 'A058470'
          AND item_code = 'M000904001'  -- 매출액
        ORDER BY date DESC
        LIMIT 8
    ''', conn)
    print('\n[4] 삼성전자(A005930) 매출액 최근 8분기')
    df4['억원'] = (df4['value'] / 1e5).round(0).astype(int)  # 천원 → 억원
    print(df4[['date', 'ticker', 'company_name', 'indicator', '억원']].to_string(index=False))

finally:
    conn.close()

korea_fs_data_from_DG 적재 검증

[1] 전체 규모
 total_rows  unique_tickers  unique_indicators   min_date   max_date
    2632071            1585                 41 2009-12-30 2026-05-29

[2] 시장/시트별 분포
market sj_div  row_cnt  ticker_cnt  indicator_cnt
    KQ     BS   544939         797             20
    KQ     CF   225097         797              8
    KQ     IS   176931         797             11
    KQ  stock   108392         797              2
    KS     BS   712238         788             20
    KS     CF   307050         788              8
    KS     IS   450256         788             11
    KS  stock   107168         788              2

[3] 중복 PK 검사: 0개 (0이어야 정상)

[4] 삼성전자(A005930) 매출액 최근 8분기
      date  ticker company_name indicator   억원
2026-03-31 A058470         리노공업   매출액(천원)  998
2025-12-30 A058470         리노공업   매출액(천원)  848
2025-09-30 A058470         리노공업   매출액(천원)  968
2025-06-30 A058470         리노공업   매출액(천원) 1125
2025-03-31 A058470         리노공업   매출액(천원)  784
2024-12-30 A058470